# Baseline v2 — PhoBERT Bi-Encoder

So sánh retrieval performance của **PhoBERT** (`vinai/phobert-base`) với **baseline v1** (`paraphrase-multilingual-MiniLM-L12-v2`).

| Metric | Baseline v1 | Baseline v2 (PhoBERT) |
|--------|-------------|------------------------|
| Recall@1 | 0.205 | ? |
| Recall@3 | 0.3447 | ? |
| Recall@5 | 0.3975 | ? |
| MRR@10 | 0.2871 | ? |

**Kiến trúc:**
- Tokenizer: AutoTokenizer (BPE, use_fast=False)
- Pooling: Mean pooling (loại trừ `<s>` và `</s>`)
- Normalize: L2
- Index: FAISS IndexFlatIP (inner product ≡ cosine sau normalize)

> ⚙️ **Yêu cầu:** torch >= 2.6.0 (do transformers 5.x enforce CVE-2025-32434).

## Cell 0 — Imports & Config

In [ ]:
import json
import csv
import numpy as np
import torch
import faiss
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

# ── Paths ──
ROOT         = Path(".")
DATA_DIR     = ROOT / "data"
EVAL_DIR     = ROOT / "outputs" / "eval"
TMP_DIR      = ROOT / "outputs" / "tmp"

TRAIN_FILE   = DATA_DIR / "train.jsonl"
DEV_FILE     = DATA_DIR / "dev.jsonl"
TRAIN_NEG    = DATA_DIR / "train_with_neg.jsonl"
EVAL_QA_FILE = EVAL_DIR / "eval_qa.jsonl"

FAISS_INDEX_V2 = TMP_DIR / "faiss_v2.index"
FAISS_MAP_V2   = TMP_DIR / "faiss_mapping_v2.jsonl"
RERANK_CSV_V2  = EVAL_DIR / "rerank_metrics_v2.csv"
RERANK_CSV_V1  = EVAL_DIR / "rerank_metrics.csv"

MODEL_NAME = "vinai/phobert-base"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
MAX_LENGTH = 256
TOP_N      = 50

print(f"torch  : {torch.__version__}")
print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_NAME}")

## Cell 1 — Utilities

In [ ]:
def load_jsonl(path, max_rows=None):
    rows, errors = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows:
                break
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                errors += 1
    if errors:
        print(f"  ⚠ {errors} malformed lines skipped in {Path(path).name}")
    return rows

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Utilities loaded ✓")

## Cell 2 — PhoBERTEncoder Class

Interface tương thích `SentenceTransformer.encode()`:
```python
encoder.encode(texts, batch_size=32, normalize_embeddings=True) → np.ndarray [N, 768]
```

In [ ]:
class PhoBERTEncoder:
    """
    Bi-Encoder wrapper cho vinai/phobert-base.
    Yêu cầu torch >= 2.6.0 (do transformers 5.x enforce CVE-2025-32434).
    """

    def __init__(self, model_name: str = "vinai/phobert-base", device: str = "cpu"):
        print(f"  Loading tokenizer: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        print(f"  Loading model...")
        self.model     = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()
        self.device    = device
        self.dim       = self.model.config.hidden_size
        print(f"  Ready | device={device} | hidden_size={self.dim}")

    @torch.no_grad()
    def _encode_batch(self, texts: list) -> np.ndarray:
        enc = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )
        ids  = enc["input_ids"].to(self.device)
        mask = enc["attention_mask"].to(self.device)

        last_h = self.model(input_ids=ids, attention_mask=mask).last_hidden_state

        # Mean pooling: bỏ <s> (pos=0) và </s> (pos=-1)
        content_h   = last_h[:, 1:-1, :]                          # [B, L-2, H]
        content_m   = mask[:, 1:-1].unsqueeze(-1).float()         # [B, L-2, 1]
        mean_pooled = (content_h * content_m).sum(1) / content_m.sum(1).clamp(min=1e-9)

        return mean_pooled.cpu().numpy().astype(np.float32)

    def encode(
        self,
        texts,
        batch_size: int = 32,
        show_progress_bar: bool = False,
        convert_to_numpy: bool = True,
        normalize_embeddings: bool = True,
    ) -> np.ndarray:
        all_vecs = []
        it = range(0, len(texts), batch_size)
        if show_progress_bar:
            it = tqdm(it, desc="PhoBERT encode")
        for start in it:
            all_vecs.append(self._encode_batch(texts[start: start + batch_size]))
        result = np.vstack(all_vecs)
        if normalize_embeddings:
            result /= np.maximum(np.linalg.norm(result, axis=1, keepdims=True), 1e-9)
        return result

print("PhoBERTEncoder defined ✓")

## Cell 3 — Load Model + Build FAISS Index

In [ ]:
# Load model
encoder = PhoBERTEncoder(MODEL_NAME, DEVICE)

# Thu thập corpus
seen_passages = {}
for f in [TRAIN_FILE, DEV_FILE, TRAIN_NEG]:
    for r in load_jsonl(f):
        p = r.get("passage", "")
        if p and p not in seen_passages:
            meta = r.get("meta", {})
            seen_passages[p] = {
                "passage":     p,
                "chunk_index": meta.get("chunk_index", -1),
                "van_ban":     meta.get("van_ban", ""),
                "chuong":      meta.get("chuong",  ""),
                "dieu":        meta.get("dieu",    ""),
                "khoan":       meta.get("khoan",   ""),
                "diem":        meta.get("diem",    ""),
            }

corpus = list(seen_passages.values())
texts  = [c["passage"] for c in corpus]
print(f"Corpus: {len(corpus)} unique passages")

# Encode
print("Encoding...")
embeddings = encoder.encode(texts, batch_size=BATCH_SIZE, show_progress_bar=True)
print(f"Shape: {embeddings.shape}")

# FAISS
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, str(FAISS_INDEX_V2))
print(f"FAISS → {FAISS_INDEX_V2}")

# Mapping
mapping = [{"faiss_id": i, **c} for i, c in enumerate(corpus)]
write_jsonl(FAISS_MAP_V2, mapping)
print(f"Mapping → {FAISS_MAP_V2} ({len(mapping)} entries)")

## Cell 4 — Evaluate Recall@K & MRR@10
Chỉ tính baseline (không rerank) để so sánh v1 vs v2.

In [ ]:
def is_hit(faiss_id, expected_citations, mapping):
    row = mapping[faiss_id]
    for ec in expected_citations:
        ci = ec.get("chunk_index", -2)
        if ci != -1 and row["chunk_index"] == ci:
            return True
        if (row["van_ban"] == ec.get("van_ban", "") and
            row["dieu"]    == ec.get("dieu",    "") and
            row["khoan"]   == ec.get("khoan",   "")):
            return True
    return False

def avg(lst):
    return round(sum(lst) / len(lst), 4) if lst else 0.0

eval_qa = load_jsonl(EVAL_QA_FILE)
print(f"Eval QA: {len(eval_qa)} questions")

results_v2 = {"R@1": [], "R@3": [], "R@5": [], "MRR@10": []}

for item in tqdm(eval_qa, desc="Evaluating v2"):
    query = item["query"]
    ec    = item["expected_citations"]

    q_emb = encoder.encode([query], normalize_embeddings=True).astype("float32")
    _, ids = index.search(q_emb, TOP_N)
    ids    = ids[0].tolist()

    for k, key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        hit = any(is_hit(i, ec, mapping) for i in ids[:k] if i >= 0)
        results_v2[key].append(1 if hit else 0)

    mrr = 0.0
    for rank, i in enumerate(ids[:10], 1):
        if i >= 0 and is_hit(i, ec, mapping):
            mrr = 1.0 / rank; break
    results_v2["MRR@10"].append(mrr)

print("\n── Baseline v2 (PhoBERT) ──")
for k, key in [("R@1","Recall@1"),("R@3","Recall@3"),("R@5","Recall@5"),("MRR@10","MRR@10")]:
    print(f"  {key}: {avg(results_v2[k]):.4f}")

## Cell 5 — So sánh v1 vs v2 & Lưu kết quả

In [ ]:
v1 = {}
if RERANK_CSV_V1.exists():
    with open(RERANK_CSV_V1, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            v1[row["metric"]] = float(row["baseline"])

v2_results = {
    "Recall@1": avg(results_v2["R@1"]),
    "Recall@3": avg(results_v2["R@3"]),
    "Recall@5": avg(results_v2["R@5"]),
    "MRR@10":   avg(results_v2["MRR@10"]),
}

print("\n" + "="*72)
print(f"  {'Metric':<10} {'Baseline v1 (multilingual)':>26} {'Baseline v2 (PhoBERT)':>22} {'Δ':>8}")
print("="*72)
for metric in ["Recall@1","Recall@3","Recall@5","MRR@10"]:
    bv1   = v1.get(metric, float("nan"))
    bv2   = v2_results[metric]
    delta = bv2 - bv1
    sign  = "+" if delta >= 0 else ""
    print(f"  {metric:<10} {bv1:>26.4f} {bv2:>22.4f} {sign}{delta:>7.4f}")
print("="*72)

rows_v2 = [
    {"metric":"Recall@1","baseline_v1":v1.get("Recall@1",""),"baseline_v2_phobert":v2_results["Recall@1"]},
    {"metric":"Recall@3","baseline_v1":v1.get("Recall@3",""),"baseline_v2_phobert":v2_results["Recall@3"]},
    {"metric":"Recall@5","baseline_v1":v1.get("Recall@5",""),"baseline_v2_phobert":v2_results["Recall@5"]},
    {"metric":"MRR@10",  "baseline_v1":v1.get("MRR@10",  ""),"baseline_v2_phobert":v2_results["MRR@10"]},
]
with open(RERANK_CSV_V2, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["metric","baseline_v1","baseline_v2_phobert"])
    w.writeheader(); w.writerows(rows_v2)

print(f"\nSaved → {RERANK_CSV_V2} ✓")